In [1]:
import torch

print("PyTorch:", torch.__version__)

PyTorch: 2.10.0


# 在 CPU 和 MPS 上的可复现 token 采样

- 当我们使用 LLM 生成文本时，通常使用 `torch.softmax` 将 logits 转换为概率，并使用 `torch.multinomial` 来采样下一个 token。
- 不幸的是，有时即使我们使用相同的随机种子，CPU 和 MPS 也可能从相同的概率中采样到不同的 token。这可能是因为每个设备使用自己的随机数生成器。
- 在本 notebook 中，我使用一个小示例来展示这种行为，并通过在 CPU 上运行采样步骤来使采样在两种情况下保持一致。
- 顺便说一下，PyTorch 不保证跨设备或版本产生完全相同的结果。有关详细信息，请参阅其 [reproducibility notes](https://docs.pytorch.org/docs/stable/notes/randomness.html)。
- 这些示例故意保持最小化。有关包含温度缩放的完整 LLM 文本生成函数，请参阅 [chapter 5](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch05/01_main-chapter-code/ch05.ipynb)。

&nbsp;
## 1) 在每个设备上采样

- 以下代码在每次调用前将随机种子硬编码为 `123`，以便我们可以使用相同的种子比较 CPU 和 MPS。

In [2]:
logits = torch.tensor(
    [[0.0, 1.0, 2.0]],
    dtype=torch.bfloat16,
)

# Convert logits to probabilities
probs = torch.softmax(logits, dim=-1)  # (batch_size, context_len)

def sample(probs):

    torch.manual_seed(123)

    # Sample from the distribution
    idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)

    return idx_next


print("CPU:", sample(probs.to("cpu")))

if torch.mps.is_available():
    print("MPS:", sample(probs.to("mps")))

if torch.cuda.is_available():
    print("CUDA:", sample(probs.to("cuda")))

CPU: tensor([[1]])
MPS: tensor([[2]], device='mps:0')


&nbsp;
## 2) 对两个设备都在 CPU 上采样

- 因此，针对上述 CPU/MPS 差异问题的一种修复或变通方法是在 `torch.multinomial` 之前调用 `probs.cpu()`。

In [3]:
def sample_on_cpu(probs):

    torch.manual_seed(123)

    # Sample from the distribution
    idx_next = torch.multinomial(probs.cpu(), num_samples=1)  # (batch_size, 1)

    return idx_next.to(probs.device)


print("CPU:", sample_on_cpu(probs.to("cpu")))

if torch.mps.is_available():
    print("MPS:", sample_on_cpu(probs.to("mps")))

if torch.cuda.is_available():
    print("CUDA:", sample_on_cpu(probs.to("cuda")))

CPU: tensor([[1]])
MPS: tensor([[1]], device='mps:0')


- 顺便说一下，这只是 MPS 和 CPU 上结果可能不同的一个方面。
- 还可能有其他原因，例如完整的 LLM，CPU 和 MPS 可能产生略微不同的 logits。这些差异会改变概率和采样的 token，因此即使在 CPU 上运行采样，生成的文本可能仍然不同。
- 此外，某些问题可能是由于某些芯片（如 M1 和 M2）中的 bug 造成的，正如 Scott Kirila 在 [Finding a Bug in Closed-Source Shader Kernels](https://scottkirila.studio.site/blog/kernel-bug) 这篇精彩的调查中所描述的那样。